In [1]:
import shutil, os

if os.path.exists("/kaggle/working/tiny-imagenet-200"):
    shutil.rmtree("/kaggle/working/tiny-imagenet-200")

if os.path.exists("/kaggle/working/tiny-imagenet-200.zip"):
    os.remove("/kaggle/working/tiny-imagenet-200.zip")

print("Clean slate")

Clean slate


In [2]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU available: True
Device: Tesla T4


In [3]:
import os, zipfile, urllib.request

DATA_DIR = "/kaggle/working/tiny-imagenet-200"

if not os.path.exists(DATA_DIR):
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    urllib.request.urlretrieve(url, "/kaggle/working/tiny-imagenet-200.zip")

    with zipfile.ZipFile("/kaggle/working/tiny-imagenet-200.zip", 'r') as zip_ref:
        zip_ref.extractall("/kaggle/working/")
print("dataset successfully loaded")

dataset successfully loaded


In [4]:
import os, shutil

val_dir = os.path.join(DATA_DIR, "val")
images_dir = os.path.join(val_dir, "images")
annotations = os.path.join(val_dir, "val_annotations.txt")

with open(annotations) as f:
    for line in f:
        img, cls = line.split()[:2]
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
        shutil.move(
            os.path.join(images_dir, img),
            os.path.join(val_dir, cls, img)
        )

shutil.rmtree(images_dir)

In [5]:
# importing all necessary packages for building model and performance metrics
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from torchvision.models import resnet50
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

In [6]:
# define data transformations on training and validation set
def get_loaders(data_dir=DATA_DIR):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(64, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4802, 0.4481, 0.3975),
            (0.2302, 0.2265, 0.2262)
        ),
        transforms.RandomErasing(p=0.25)
    ])

    val_tf = transforms.Compose([
        transforms.Resize(64),
        transforms.CenterCrop(64),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4802, 0.4481, 0.3975),
            (0.2302, 0.2265, 0.2262)
        ),
    ])

    train_ds = torchvision.datasets.ImageFolder(
        os.path.join(data_dir, "train"),
        transform=train_tf
    )
    val_ds = torchvision.datasets.ImageFolder(
        os.path.join(data_dir, "val"),
        transform=val_tf
    )

    train_loader = DataLoader(
        train_ds, batch_size=128, shuffle=True,
        num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=256, shuffle=False,
        num_workers=2
    )

    return train_loader, val_loader

In [7]:
# perform data augmentation
def rand_bbox(size, lam):
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)

    cx, cy = np.random.randint(W), np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2


def cutmix(x, y, alpha=1.0):
    indices = torch.randperm(x.size(0)).to(x.device)
    lam = np.random.beta(alpha, alpha)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[indices, :, bbx1:bbx2, bby1:bby2]

    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size(-1) * x.size(-2)))

    return x, y, y[indices], lam

In [8]:
# create evaluation fcn
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return 100 * correct / total

In [12]:
# create cnn model
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.features = nn.Sequential(
            # block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # block 3
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [13]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print(torch.cuda.is_available())

train_loader, val_loader = get_loaders()

model = CNN().to(device)

# define optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=.01,
    momentum=.9,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50
)

criterion = nn.CrossEntropyLoss()

from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()

import time
import psutil
import torch

def get_model_size(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024**2  # MB

def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_gpu_usage():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2  # MB
    return 0

torch.backends.cudnn.benchmark = True

def train(model, train_loader, val_loader, device):
    best_acc = 0
    patience = 20
    counter = 0

    total_start_time = time.time()

    print("\n===== TRAINING START =====")

    for epoch in range(50):
        epoch_start = time.time()

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            # apply cutmix before fwd pass
            if np.random.rand() < 0.5:
                images, targets_a, targets_b, lam = cutmix(images, labels)
                outputs = model(images)
                loss = lam * criterion(outputs, targets_a) + \
                       (1 - lam) * criterion(outputs, targets_b)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        epoch_time = time.time() - epoch_start
        val_acc = evaluate(model, val_loader, device)

        # early stopping logic
        if best_acc < val_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print("Early stopping")
            break

        # basic logging
        print(f"\nEpoch {epoch+1:02d}")
        print(f"Loss: {total_loss:.2f}")
        print(f"Val Acc: {val_acc:.2f}%")
        print(f"Epoch Time: {epoch_time:.2f}s")

        # print avg cpu/gpu usage every 10 epochs
        if (epoch + 1) % 10 == 0:
            cpu_usage = psutil.cpu_percent()
            gpu_usage = torch.cuda.memory_allocated() / 1024**2
            print(f"Avg CPU Usage: {cpu_usage:.2f}%")
            print(f"Avg GPU Memory: {gpu_usage:.2f} MB")

    total_time = time.time() - total_start_time

    print("\n===== TRAINING COMPLETE =====")
    print(f"Total Training Time: {total_time/60:.2f} minutes")
    print(f"Best Validation Accuracy: {best_acc:.2f}%")

    # print model size at the end
    print(f"Model Size: {get_model_size(model):.2f} MB")

cuda
True


/tmp/ipykernel_55/77694988.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [14]:
# make sure it is using GPU
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# train model
torch.cuda.amp # speed it up
train(model, train_loader, val_loader, device)

CUDA: True
Device: Tesla T4

===== TRAINING START =====

Epoch 01
Loss: 3995.09
Val Acc: 4.55%
Epoch Time: 97.86s

Epoch 02
Loss: 3812.53
Val Acc: 6.96%
Epoch Time: 92.89s

Epoch 03
Loss: 3744.47
Val Acc: 8.95%
Epoch Time: 93.11s

Epoch 04
Loss: 3672.23
Val Acc: 9.51%
Epoch Time: 92.97s

Epoch 05
Loss: 3604.36
Val Acc: 10.20%
Epoch Time: 91.98s

Epoch 06
Loss: 3562.90
Val Acc: 11.07%
Epoch Time: 92.58s

Epoch 07
Loss: 3511.75
Val Acc: 14.63%
Epoch Time: 92.20s

Epoch 08
Loss: 3469.52
Val Acc: 13.78%
Epoch Time: 92.49s

Epoch 09
Loss: 3422.44
Val Acc: 17.47%
Epoch Time: 92.79s

Epoch 10
Loss: 3408.25
Val Acc: 16.02%
Epoch Time: 92.83s
Avg CPU Usage: 49.60%
Avg GPU Memory: 121.84 MB

Epoch 11
Loss: 3364.66
Val Acc: 15.88%
Epoch Time: 91.69s

Epoch 12
Loss: 3318.59
Val Acc: 16.92%
Epoch Time: 91.90s

Epoch 13
Loss: 3305.28
Val Acc: 19.28%
Epoch Time: 92.12s

Epoch 14
Loss: 3255.66
Val Acc: 19.56%
Epoch Time: 93.07s

Epoch 15
Loss: 3227.52
Val Acc: 21.71%
Epoch Time: 93.51s

Epoch 16
Loss:

In [9]:
# define better CNN (more layers w/ dropout)
import torch
import torch.nn as nn

class BetterCNN(nn.Module):
    def __init__(self, num_classes=200):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Dropout2d(0.2),   
            nn.MaxPool2d(2),     

            # Block 5
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Dropout2d(0.2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),  
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [10]:
# training function for betterCNN

import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print(torch.cuda.is_available())

train_loader, val_loader = get_loaders()

better_model = BetterCNN().to(device)

# define optimizer
better_CNN_optimizer = torch.optim.SGD(
    better_model.parameters(),
    lr=0.004,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    better_CNN_optimizer, T_max=50
)

better_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()

import time
import psutil
import torch

# functions for tracking time, model size, GPU/CPU usage, etc.
def get_model_size(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024**2  # MB

def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_gpu_usage():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2  # MB
    return 0

torch.backends.cudnn.benchmark = True

def train_with_tracking(model, train_loader, val_loader, device):
    best_acc = 0
    total_start_time = time.time()

    print("\n===== TRAINING START =====")

    for epoch in range(50):
        epoch_start = time.time()

        model.train()
        total_loss = 0

        cpu_usage_epoch = []
        gpu_usage_epoch = []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
        
            better_CNN_optimizer.zero_grad()
        
            if np.random.rand() < 0.5:
                images, targets_a, targets_b, lam = cutmix(images, labels)
            else:
                targets_a, targets_b, lam = labels, labels, 1.0
        
            with autocast():
                outputs = model(images)
                loss = lam * better_criterion(outputs, targets_a) + \
                       (1 - lam) * better_criterion(outputs, targets_b)
        
            scaler.scale(loss).backward()
            scaler.step(better_CNN_optimizer)
            scaler.update()

            total_loss += loss.item()

            # track CPU/GPU usage
            cpu_usage_epoch.append(psutil.cpu_percent(interval=None))
            if torch.cuda.is_available():
                gpu_usage_epoch.append(torch.cuda.memory_allocated() / 1024**2)

        scheduler.step()

        epoch_time = time.time() - epoch_start
        val_acc = evaluate(model, val_loader, device)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "/kaggle/working/best_model.pth")

        print(f"\nEpoch {epoch+1:02d}")
        print(f"Loss: {total_loss:.2f}")
        print(f"Val Acc: {val_acc:.2f}%")
        print(f"Epoch Time: {epoch_time:.2f}s")

        # display avg CPU/GPU usage every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Avg CPU Usage: {sum(cpu_usage_epoch)/len(cpu_usage_epoch):.2f}%")
            if gpu_usage_epoch:
                print(f"Avg GPU Memory: {sum(gpu_usage_epoch)/len(gpu_usage_epoch):.2f} MB")

    total_time = time.time() - total_start_time

    print("\n===== TRAINING COMPLETE =====")
    print(f"Total Training Time: {total_time/60:.2f} minutes")
    print(f"Best Validation Accuracy: {best_acc:.2f}%")

    # print model size after training completes
    print(f"Final Model Size: {get_model_size(model):.2f} MB")


cuda
True


/tmp/ipykernel_57/839676726.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [11]:
# make sure it is using GPU
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# train model
torch.cuda.amp # speed it up
train_with_tracking(better_model, train_loader, val_loader, device)

CUDA: True
Device: Tesla T4

===== TRAINING START =====


/tmp/ipykernel_57/839676726.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Epoch 01
Loss: 4084.18
Val Acc: 3.27%
Epoch Time: 121.83s

Epoch 02
Loss: 3913.74
Val Acc: 5.29%
Epoch Time: 91.91s

Epoch 03
Loss: 3802.11
Val Acc: 9.55%
Epoch Time: 93.41s

Epoch 04
Loss: 3699.70
Val Acc: 9.24%
Epoch Time: 93.79s

Epoch 05
Loss: 3640.39
Val Acc: 12.04%
Epoch Time: 94.63s

Epoch 06
Loss: 3587.51
Val Acc: 13.58%
Epoch Time: 93.75s

Epoch 07
Loss: 3529.39
Val Acc: 14.43%
Epoch Time: 94.90s

Epoch 08
Loss: 3499.97
Val Acc: 16.89%
Epoch Time: 95.35s

Epoch 09
Loss: 3484.74
Val Acc: 17.63%
Epoch Time: 92.63s

Epoch 10
Loss: 3435.60
Val Acc: 18.56%
Epoch Time: 94.17s
Avg CPU Usage: 64.93%
Avg GPU Memory: 172.42 MB

Epoch 11
Loss: 3382.23
Val Acc: 19.48%
Epoch Time: 93.33s

Epoch 12
Loss: 3381.05
Val Acc: 21.30%
Epoch Time: 93.92s

Epoch 13
Loss: 3348.66
Val Acc: 22.93%
Epoch Time: 94.46s

Epoch 14
Loss: 3321.47
Val Acc: 22.97%
Epoch Time: 95.33s

Epoch 15
Loss: 3326.52
Val Acc: 25.00%
Epoch Time: 94.73s

Epoch 16
Loss: 3312.40
Val Acc: 23.64%
Epoch Time: 94.43s

Epoch 17
L

In [13]:
# converting fp32 model to fp16
def convert_to_fp16():
    model = BetterCNN()
    model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))

    model = model.to(device)
    model.eval()

    # convert weights to FP16
    model.half()

    print("\nConverted model to FP16")

    return model

In [14]:
# evaluation fcn for fp16 model
def evaluate_fp16(model, loader):
    model.eval()

    correct = total = 0
    cpu_usage = []
    gpu_usage = []

    start = time.time()

    with torch.no_grad():
        for x, y in loader:
            # halving inputs so they are also fp16
            x = x.to(device).half()
            y = y.to(device)

            out = model(x)
            pred = out.argmax(1)

            correct += (pred == y).sum().item()
            total += y.size(0)

            cpu_usage.append(get_cpu_usage())
            if torch.cuda.is_available():
                gpu_usage.append(get_gpu_usage())

    total_time = time.time() - start
    acc = 100 * correct / total

    print("\n===== FP16 EVALUATION =====")
    print(f"Accuracy: {acc:.2f}%")
    print(f"Inference Time: {total_time:.2f}s")
    print(f"Model Size: {get_model_size(model):.2f} MB")
    print(f"Avg CPU Usage: {sum(cpu_usage)/len(cpu_usage):.2f}%")

    if gpu_usage:
        print(f"Avg GPU Memory: {sum(gpu_usage)/len(gpu_usage):.2f} MB")

    return acc

In [15]:
# convert and evaluate fp16 model
fp16_model = convert_to_fp16()

# evaluate
evaluate_fp16(fp16_model, val_loader)


Converted model to FP16

===== FP16 EVALUATION =====
Accuracy: 36.86%
Inference Time: 3.70s
Model Size: 24.67 MB
Avg CPU Usage: 60.33%
Avg GPU Memory: 196.51 MB


36.86

In [18]:
# fp16 model & training (QAT, didn't end up working)

import time
import psutil
import torch
from torch.cuda.amp import GradScaler, autocast

fp16_model = BetterCNN().to(device)

# optimizer for fp16
fp16_optimizer = torch.optim.SGD(
    fp16_model.parameters(),
    lr=0.005,              
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    fp16_optimizer, T_max=50
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

scaler = GradScaler()

def get_model_size(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024**2 

def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_gpu_usage():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2 
    return 0

# train fcn for fp16
def train_fp16(model, train_loader, val_loader, device):
    best_acc = 0
    total_start = time.time()

    print("\n===== FP16 TRAINING START =====")

    for epoch in range(50):
        epoch_start = time.time()

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            fp16_optimizer.zero_grad()

            # cutmix
            if np.random.rand() < 0.5:
                images, targets_a, targets_b, lam = cutmix(images, labels)
            else:
                targets_a, targets_b, lam = labels, labels, 1.0

            # fwd pass
            with autocast():
                outputs = model(images)
                loss = lam * criterion(outputs, targets_a) + \
                       (1 - lam) * criterion(outputs, targets_b)

            scaler.scale(loss).backward()
            scaler.step(fp16_optimizer)
            scaler.update()

            total_loss += loss.item()

        scheduler.step()

        epoch_time = time.time() - epoch_start
        val_acc = evaluate(model, val_loader, device)

        # save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "/kaggle/working/best_fp16_model.pth")

        print(f"\n[FP16] Epoch {epoch+1:02d}")
        print(f"Loss: {total_loss:.2f}")
        print(f"Val Acc: {val_acc:.2f}%")
        print(f"Epoch Time: {epoch_time:.2f}s")

        # display usage every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"CPU Usage: {get_cpu_usage():.2f}%")
            print(f"GPU Memory: {get_gpu_usage():.2f} MB")

    total_time = time.time() - total_start

    print("\n===== FP16 TRAINING COMPLETE =====")
    print(f"Total Time: {total_time/60:.2f} minutes")
    print(f"Best Accuracy: {best_acc:.2f}%")
    print(f"Model Size: {get_model_size(model):.2f} MB")

/tmp/ipykernel_55/3215988266.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [19]:
# train and run fp16 model
torch.cuda.amp
train_fp16(fp16_model, train_loader, val_loader, device)


===== FP16 TRAINING START =====


/tmp/ipykernel_55/3215988266.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



[FP16] Epoch 01
Loss: 4072.95
Val Acc: 3.32%
Epoch Time: 90.88s

[FP16] Epoch 02
Loss: 3901.04
Val Acc: 5.53%
Epoch Time: 90.93s

[FP16] Epoch 03
Loss: 3772.12
Val Acc: 9.08%
Epoch Time: 91.26s

[FP16] Epoch 04
Loss: 3687.97
Val Acc: 10.26%
Epoch Time: 90.74s

[FP16] Epoch 05
Loss: 3605.33
Val Acc: 12.32%
Epoch Time: 90.69s

[FP16] Epoch 06
Loss: 3563.58
Val Acc: 13.84%
Epoch Time: 90.45s

[FP16] Epoch 07
Loss: 3517.22
Val Acc: 15.63%
Epoch Time: 90.53s

[FP16] Epoch 08
Loss: 3469.97
Val Acc: 17.01%
Epoch Time: 90.28s

[FP16] Epoch 09
Loss: 3454.78
Val Acc: 17.37%
Epoch Time: 90.23s

[FP16] Epoch 10
Loss: 3408.36
Val Acc: 18.99%
Epoch Time: 90.62s
CPU Usage: 21.60%
GPU Memory: 420.58 MB

[FP16] Epoch 11
Loss: 3373.99
Val Acc: 21.89%
Epoch Time: 89.73s

[FP16] Epoch 12
Loss: 3351.04
Val Acc: 22.61%
Epoch Time: 90.15s

[FP16] Epoch 13
Loss: 3345.28
Val Acc: 21.47%
Epoch Time: 90.98s

[FP16] Epoch 14
Loss: 3314.47
Val Acc: 24.69%
Epoch Time: 89.64s

[FP16] Epoch 15
Loss: 3269.06
Val Acc:

In [16]:
# 8bit quantization - load trained model
model = BetterCNN()
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))

int8_start = time.time()

model_cpu = model.to("cpu")
model_cpu.eval()

quantized_model = torch.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

acc = evaluate(quantized_model, val_loader, "cpu")
print(f"INT8 Accuracy: {acc}%")
print(f"INT8 Size: {get_model_size(quantized_model):.2f} MB")
print(f"INT8 CPU Usage: {get_cpu_usage():.2f}%")
print(f"INT8 GPU Usage: {get_gpu_usage():.2f} MB")
print("INT8 Quantization Time: ", time.time() - int8_start)

/tmp/ipykernel_57/3027011869.py:10: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


INT8 Accuracy: 36.84%
INT8 Size: 14.94 MB
INT8 CPU Usage: 19.80%
INT8 GPU Usage: 190.55 MB
INT8 Quantization Time:  147.58596086502075
